In [0]:
# Initialize Database
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.bronze")
spark.sql(f"USE {catalog}.bronze")

In [0]:
%pip install librosa

In [0]:
# Create a Volume to store checkpoints and other non-tabular files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.checkpoints")
print(f"Volume created: /Volumes/{catalog}/bronze/checkpoints")

In [0]:
# -------------------------------------------------------------------------
# BRONZE LAYER: RAW STREAMING INGESTION (EXTERNAL LOCATION VERSION)
# -------------------------------------------------------------------------

import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_date, 
    year, month
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    FloatType, TimestampType
)

dbutils.widgets.text("kafka_bootstrap_servers", "0.tcp.ap.ngrok.io:17438", "Kafka Ngrok URL")
dbutils.widgets.text("kafka_topic", "audio_uploads", "Kafka Topic")
dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")

kafka_url = dbutils.widgets.get("kafka_bootstrap_servers")
kafka_topic = dbutils.widgets.get("kafka_topic")
catalog = dbutils.widgets.get("catalog_name")

checkpoint_path = f"/Volumes/{catalog}/bronze/checkpoints/audio_events_stream"
dbutils.fs.mkdirs(checkpoint_path)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.bronze")
spark.sql(f"USE {catalog}.bronze")
print(f"Using {catalog}.bronze managed by Unity Catalog.")

kafka_message_schema = StructType([
    StructField("track_id", IntegerType(), False),
    StructField("metadata", StructType([
        StructField("title", StringType(), True),
        StructField("duration", FloatType(), True),
        StructField("artist_name", StringType(), True),
        StructField("genre_top", StringType(), True)
    ]), True),
    StructField("s3_uri", StringType(), False),
    StructField("timestamp", StringType(), True) 
])

kafka_stream_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_url)
    .option("subscribe", kafka_topic)
    # Change 'latest' to 'earliest' to catch messages already in the queue
    .option("startingOffsets", "earliest") 
    .load())

bronze_df = (kafka_stream_df
    .selectExpr("CAST(value AS STRING) as json_payload", "timestamp as kafka_time")
    .select(from_json(col("json_payload"), kafka_message_schema).alias("data"), "kafka_time")
    .select(
        "data.track_id",
        "data.metadata.*",
        "data.s3_uri",
        col("data.timestamp").cast(TimestampType()).alias("event_time"),
        "kafka_time",
        current_timestamp().alias("ingested_at")
    )
    .withColumn("event_date", to_date(col("event_time")))
)

table_name = f"{catalog}.bronze.audio_events"

print(f"Starting Batch Ingestion: Kafka -> {table_name}")

# 'availableNow=True' processes all available data and then stops.
query = (bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .partitionBy("event_date")
    .trigger(availableNow=True)  # fix for Serverless.
    .toTable(table_name))

print("Processing batch... please wait.")
query.awaitTermination() 

if not query.isActive:
    print(f"Batch Complete! Data is now in {table_name}")

In [0]:
display(spark.table("fma_catalog.bronze.audio_events"))

In [0]:
spark.table("fma_catalog.bronze.audio_events").printSchema()

In [0]:
# Wipe the old checkpoint so Spark looks at the 'earliest' offsets again
dbutils.fs.rm(checkpoint_path, recurse=True)
print(f"CheckApoint cleared at {checkpoint_path}")